# Modeling

This notebook tests multiple classification models for Titanic survival prediction. The goal of this phase is to build baseline models, apply hyperparameter tuning, and identify the strongest candidate models for deeper evaluation.

The models tested include Logistic Regression, Support Vector Machine, K-Nearest Neighbors, Random Forest, and XGBoost. Model performance was compared using classification reports, confusion matrices, recall, precision, and F1-score.

The strongest candidate models from this phase were Logistic Regression and Support Vector Machine. These models were carried forward into the model evaluation notebook for additional testing, including overfitting checks, cross-validation, calibration, threshold tuning, and error analysis.

Important note: GridSearchCV and HalvingGridSearchCV results in this notebook are treated as candidate parameter selections, not final model decisions. Final parameters were selected later after reviewing generalization and stability in the model evaluation phase.

In [ ]:
# import both datasets with significant features and target value 

from pathlib import Path # access to working directory 
import pandas as pd # data manipulation 

# current working directory 
dir = Path.cwd()

# look into Data and find linear.csv and non_linear.csv
data_path = dir.parent / 'Data' / 'linear.csv'
data_path2 = dir.parent / 'Data' / 'non_linear.csv'
target_path = dir.parent / 'Data' / 'target.csv'

# load data path into csv
linear = pd.read_csv(data_path)
non_linear = pd.read_csv(data_path2)
target = pd.read_csv(target_path)


In [ ]:
# training phase for simpler models
# import sklearn train test split 
from sklearn.model_selection import train_test_split

# target value
Y = target['Survived']

# covariate variables (linear)
X = linear[['Pclass', 'Age', 'SibSp', 'Sex_female', 'Cabin_encoded']]

# 60/40 train/test split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.3, shuffle= True, random_state = 42)


In [ ]:
# Modeling Phase (Classification)
# import sklearn numeric performance indicators for downstream modeling workflow 
from sklearn.metrics import confusion_matrix, classification_report 


 Logistic Regression

Logistic Regression was tested as the primary linear classification model. This model is useful as a baseline because it performs well on structured data and provides more interpretable prediction behavior than many non-linear models.

The baseline Logistic Regression model produced strong classification results, with recall around 81%, precision around 75%, and an F1-score around 78%. This showed that the model was effective at identifying survivors while maintaining a reasonable balance between precision and recall.

GridSearchCV was then used to test multiple solver, penalty, and regularization settings. The tuned results were treated as candidate parameters for later evaluation rather than final model selections, since later testing was needed to confirm whether the model generalized well.

In [ ]:
# basline model (logistic regression) 

from sklearn.linear_model import LogisticRegression

# begin instantiantion process
lr = LogisticRegression(max_iter= 100)

# fit data on model
lr.fit(X_train, Y_train)

# predictions 
lr_pred = lr.predict(X_test)

# performance metrrics
print(f'Classification Report {classification_report(Y_test, lr_pred)}')
print(f'Confusion matric {confusion_matrix(Y_test, lr_pred)}')


In [ ]:
# Let's find the best hyperparameter settings for the model 
from sklearn.model_selection import GridSearchCV

# parameters
params = [ 
    {
        'solver': ['liblinear'], 
        'penalty': ['l1', 'l2'], 
        'C': [0.001, 0.01, 0.1, 1, 10, 50, 100],
        'fit_intercept': [True], 
    
    }, 
    { 
        'solver': ['lbfgs'], 
        'penalty': ['l2'], # lbfgs cannot do L1
        'C': [0.001, 0.1, 1, 10, 100],
        'fit_intercept': [True]

    }, 
    {
        'solver': ['saga'], 
        'penalty': ['elasticnet'],
        'C': [0.01, 0.1, 1, 10, 100], 
        'l1_ratio': [0.5], # required for elasticnet 
        'fit_intercept': [True]
    }
]


# instantiantion process 
lr_best = GridSearchCV(estimator= lr, param_grid= params, cv = 10, verbose= 5, n_jobs= -1, scoring = 'recall')

# fit data on model
lr_best.fit(X_train, Y_train)

# predictions
lr_pred = lr_best.predict(X_test)

# performance metrics 
print(f'Classification Report C{classification_report(Y_test, lr_pred)}')
print(f'Confusion Matrix {confusion_matrix(Y_test, lr_pred)}')
print(f'Best settings {lr_best.best_estimator_}')

# Support Vector Machine

Support Vector Machine was tested as a non-linear classification model using the engineered non-linear feature set. SVM was included because it can capture more complex decision boundaries than Logistic Regression.

The baseline SVM model produced competitive results, with recall around 80%, precision around 76%, and an F1-score around 78%. This made it one of the strongest candidate models during the modeling phase.

During tuning, the full GridSearchCV process was computationally expensive, so HalvingGridSearchCV was used to reduce runtime by evaluating candidate parameters on progressively larger subsets of data. The tuned SVM settings were treated as candidate parameters and carried forward into the evaluation phase for deeper generalization testing.

In [ ]:
# training phase (non linearity)
from sklearn.model_selection import train_test_split 

# target variable 
Y = target['Survived']

# inputs 
X = non_linear.copy()

# 70/30 split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.3, random_state = 42) 


In [ ]:
from sklearn.svm import SVC # support vector model 
# instantiation process
svm = SVC()

# fit the data on the model
svm.fit(X_train, Y_train)

# predictios
Y_predictions = svm.predict(X_test)

# predictions results
Y_predictions

# Evaluate model performance results 
print(f'Classification Report {classification_report(Y_test, Y_predictions)}')
print(f'Confusion Matrix {confusion_matrix(Y_test, Y_predictions)}')

In [ ]:
# find the best settings for the SVM model 
from sklearn.experimental import enable_halving_search_cv # required!
from sklearn.model_selection import HalvingGridSearchCV


# parameters
params = [
    {
        'kernel': ['linear'], 
        'C': [0.1, 1, 10, 100]
        
    },

    {
        'kernel': ['rbf'], 
        'C': [0.1, 1, 10, 100], 
        'gamma': [0.0001, 0.001, 0.1, 1]
     
    },
    {
        'kernel': ['poly'], 
        'degree': [2, 3, 4], 
        'C': [0.1, 1, 10, 100] 
    
    }
]
# Grid Search settings 
svm_best = HalvingGridSearchCV(estimator = svm, param_grid = params, factor = 6,  resource = 'n_samples', n_jobs = -1, verbose = 5, scoring = 'recall', cv= 10)
# fit data onto model
svm_best.fit(X_train, Y_train)

# predictions
svm_pred = svm_best.predict(X_test)

# performance result 
print(f'Classification Report: {classification_report(Y_test, svm_pred)}')
print(f'confusion Matrix: {confusion_matrix(Y_test, svm_pred)}')
print(f'Best Settings: {svm_best.best_estimator_}')

# K-Nearest Neighbors

K-Nearest Neighbors was tested as a distance-based classification model using the non-linear feature set. KNN can capture local patterns in the data but is sensitive to feature scaling, noisy variables, and the choice of neighbors.

The baseline KNN model produced recall around 84%, precision around 70%, and an F1-score around 76%. Although recall was strong, the lower precision suggested that the model made more false positive survival predictions.

After hyperparameter tuning, performance did not improve enough to justify moving KNN forward as a final candidate. Because of this, KNN was not selected for deeper downstream evaluation.



In [ ]:
# (KNN) uses distances to classify which class a data point belongs in 

from sklearn.neighbors import KNeighborsClassifier 

# instantiation process
KNN = KNeighborsClassifier(n_neighbors=5)

# fit the data on the model 
KNN.fit(X_train, Y_train)

# predictions
KNN_pred = KNN.predict(X_test)

# performance results 
print(f'Classification Report: {classification_report(Y_test, KNN_pred)}')
print(f'Confusion Matric: {confusion_matrix(Y_test, KNN_pred)}')

In [ ]:
# Find the best optimal settings for K-Nearest Neighbors 
# import gridsearchcv to locate the best paramaters for the model
from sklearn.model_selection import GridSearchCV

params = {'n_neighbors': [3, 5, 7, 8, 11, 15, 21], 
          'weights': ['uniform', 'distance'], 
          'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 
          'metric': ['euclidean', 'manhattan']}

# instantiation process
KNN_best = GridSearchCV(estimator = KNN, param_grid= params, cv = 10, verbose = 5, scoring = 'recall')

# fit data on the model 
KNN_best.fit(X_train, Y_train)

# predictions 
KNN_pred = KNN_best.predict(X_test)

# performance results
print(f'Classification Report:{classification_report(Y_test, KNN_pred)}')
print(f'Best Settings: {(KNN_best.best_estimator_)}')

# Random Forest Classification

Random Forest was tested as an ensemble tree-based classification model. This model can capture non-linear relationships and feature interactions without requiring the same assumptions as linear models.

The model produced recall around 71%, precision around 79%, and an F1-score around 73%. While precision was reasonable, recall was weaker than Logistic Regression, SVM, and KNN, meaning the model missed more actual survivors.

HalvingGridSearchCV was used during tuning to reduce the runtime of the parameter search. Although tuning improved performance slightly, the results were not strong enough to move Random Forest into the final evaluation phase.

In [ ]:
# Random Forest Classification 
from sklearn.ensemble import RandomForestClassifier 

# instantiation process
rf = RandomForestClassifier(n_estimators= 400)
# fit data onto model 
rf.fit(X_train, Y_train)

# predictions
rf_pred = rf.predict(X_test)

# peformance evaluation
print(f'Classification Repor: {classification_report(Y_test, rf_pred)}')
print(f'Confusion Matrix: {confusion_matrix(Y_test, rf_pred)}')

In [ ]:
# Find best parameters for the model 
# params 
params = {'n_estimators': [300, 400, 500, 800], 
          'max_depth': [None, 10, 20, 30], 
          'min_samples_split': [2, 5, 10], 
          'max_features': ['sqrt', 'log2'],
          'random_state': [42], 
          'n_jobs': [-1]}

# instantation process 
rf_best = HalvingGridSearchCV(estimator= rf, param_grid= params, cv = 10, verbose = 5, n_jobs = -1, factor = 3, scoring = 'recall')
# fit data onto model 
rf_best.fit(X_train, Y_train)

# predicitions
rf_pred = rf_best.predict(X_test)

# performance summary 
print(f'Classification Report: {classification_report(Y_test, rf_pred)}')
print(f'Confusion Matrix: {confusion_matrix(Y_test, rf_pred)}')
print(f'Best Settings: {rf_best.best_estimator_}')

# XGBoost Classification

XGBoost was tested as a boosted tree-based classification model. This model can perform well on structured datasets by combining multiple weak learners into a stronger predictive model.

The XGBoost model produced recall around 69%, precision around 75%, and an F1-score around 72%. These results were weaker than the strongest candidate models, especially in recall.

Because the model underperformed compared with Logistic Regression and SVM, XGBoost was not selected for downstream evaluation.

In [ ]:
# Let's use a boosted algorithm to capture omore accuacry and performance 

# import package 
import xgboost as xgb 

# intialize xgboostclassifier
xgb = xgb.XGBClassifier(n_estimators = 100)
# fit data on boosted algo
xgb.fit(X_train, Y_train)

# predictions
xgb_pred = xgb.predict(X_test)

# performance metrics 
print(f'Classification Report {classification_report(Y_test, xgb_pred)}')
print(f'Confusin Matrix {confusion_matrix(Y_test, xgb_pred)}')


In [ ]:
# Let's find the best settings for the boosted algorithm

# parameters 
params = {'n_estimators': [100, 300, 500],
          'max_depth': [3, 5, 7, 9], 
          'min_features' 
          'subsample': [0.6, 0.8, 1.0], 
          'colsample_bytree': [0.6, 0.8, 1.0], 
          'gamma': [0, 0.1, 0.2]
}

# intitalize gridsearch 
xgb_best = GridSearchCV(estimator= xgb, param_grid= params, cv = 10, verbose = 5, n_jobs =-1) 

# fit data on model
xgb_best.fit(X_train, Y_train)

# predcitions
xgb_pred = xgb.predict(X_test)

# evaluation metrics 
print(f'Classification Report {classification_report(Y_test, xgb_pred)}')
print(f'Confusion Matirx {confusion_matrix(Y_test, xgb_pred)}')
print(f'Best settings {xgb_best.best_estimator_}')


# Grid Search Result Review

GridSearchCV was used during the model buildout phase to identify the strongest hyperparameter combinations based on cross-validation performance. However, the best parameters returned by GridSearchCV were treated as candidate parameters rather than final model selections.

During the later model evaluation phase, some of these parameter combinations showed signs of overfitting or unstable generalization when compared across training performance, testing performance, cross-validation behavior, and error analysis.

As a result, the final model parameters were selected after additional evaluation rather than relying only on the original GridSearchCV output.